In [11]:
!pip install flask pyngrok transformers torch newspaper3k lxml[html5] lxml_html_clean

In [12]:
!mkdir -p templates
!mkdir -p static
!mkdir -p uploads

# FLASK

In [13]:
%%writefile app.py
from flask import Flask, render_template, request
from transformers import pipeline
from newspaper import Article
import os

app = Flask(__name__)

# ------------------- Load Summarization Model -------------------
summarizer = pipeline("summarization", model="facebook/bart-large-cnn")

def summarize_text_or_url(input_text=None, input_url=None):
    if input_url:
        article = Article(input_url)
        article.download()
        article.parse()
        text = article.text
    elif input_text:
        text = input_text
    else:
        return "❗ Please provide text or a URL."

    summary = summarizer(text, max_length=150, min_length=50, do_sample=False)
    return summary[0]['summary_text']


@app.route("/", methods=["GET", "POST"])
def home():
    summary = ""
    input_text = ""
    input_url = ""

    if request.method == "POST":
        input_text = request.form.get("input_text")
        input_url = request.form.get("input_url")

        summary = summarize_text_or_url(input_text=input_text, input_url=input_url)

    return render_template("index.html", summary=summary, input_text=input_text, input_url=input_url)


if __name__ == "__main__":
    app.run(host="0.0.0.0", port=8000, debug=False)

Overwriting app.py


# HTML

In [14]:
%%writefile templates/index.html
<!DOCTYPE html>
<html>
<head>
    <title>📰 Text & URL Summarizer</title>
    <link rel="stylesheet" href="{{ url_for('static', filename='style.css') }}">
</head>
<body>
<div class="container">
    <h1>📰 Text & URL Summarizer</h1>

    <div class="card">
        <h2>Enter Text</h2>
        <form method="post">
            <textarea name="input_text" placeholder="Paste text here..." rows="6">{{ input_text }}</textarea>
            <h3>OR</h3>
            <input type="url" name="input_url" placeholder="Enter article URL" value="{{ input_url }}">
            <button type="submit">Summarize 🚀</button>
        </form>
    </div>

    {% if summary %}
    <div class="card">
        <h2>Summary</h2>
        <p>{{ summary }}</p>
    </div>
    {% endif %}
</div>
</body>
</html>


Overwriting templates/index.html


# CSS

In [15]:

%%writefile static/style.css
body {
    font-family: 'Segoe UI', sans-serif;
    background: linear-gradient(135deg, #0f2027, #203a43, #2c5364);
    color: white;
    display: flex;
    justify-content: center;
    align-items: center;
    height: 100vh;
    margin: 0;
}
.container {
    text-align: center;
    width: 60%;
}
h1, h2, h3 {
    margin-bottom: 15px;
}
.card {
    background: rgba(255, 255, 255, 0.1);
    padding: 20px;
    border-radius: 12px;
    box-shadow: 0 0 15px rgba(0,0,0,0.4);
    margin-bottom: 20px;
}
textarea, input, button {
    width: 90%;
    margin: 10px 0;
    padding: 10px;
    border-radius: 8px;
    border: none;
}
textarea {
    resize: vertical;
}
button {
    background: #FFD700;
    cursor: pointer;
    font-weight: bold;
}
button:hover {
    background: #FFA500;
}

Overwriting static/style.css


In [16]:
# Kill any previous Flask/ngrok processes
!pkill -f flask || echo "No flask running"
!pkill -f ngrok || echo "No ngrok running"

^C
^C


In [17]:
!lsof -i : 8000

lsof: unacceptable port specification in: -i :
lsof 4.93.2
 latest revision: https://github.com/lsof-org/lsof
 latest FAQ: https://github.com/lsof-org/lsof/blob/master/00FAQ
 latest (non-formatted) man page: https://github.com/lsof-org/lsof/blob/master/Lsof.8
 usage: [-?abhKlnNoOPRtUvVX] [+|-c c] [+|-d s] [+D D] [+|-E] [+|-e s] [+|-f[gG]]
 [-F [f]] [-g [s]] [-i [i]] [+|-L [l]] [+m [m]] [+|-M] [-o [o]] [-p s]
 [+|-r [t]] [-s [p:s]] [-S [t]] [-T [t]] [-u s] [+|-w] [-x [fl]] [--] [names]
Use the ``-h'' option to get more help information.


In [8]:

!kill -9 1144

/bin/bash: line 1: kill: (1144) - No such process


In [9]:
!nohup python app.py > flask.log 2>&1 &

In [18]:

# Run Flask in background
!nohup python app.py > flask.log 2>&1 &

# Start ngrok tunnel
from pyngrok import ngrok, conf
conf.get_default().auth_token = "use_ngrok _token"
public_url = ngrok.connect(8000)
print("🌍 Public URL:", public_url)

# View logs
!sleep 3 && tail -n 20 flask.log



🌍 Public URL: NgrokTunnel: "https://jamarion-awninged-colleen.ngrok-free.dev" -> "http://localhost:8000"
